# Body Measurement Visualization (Solution)

This notebook provides a comprehensive visualization of the body measurement extraction process, including video preprocessing, body detection, 3D point cloud generation, 3D model reconstruction, and measurement calculation with interactive plots and detailed analysis.

## 1. Setup and Configuration

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import open3d as o3d
from body_measurement import BodyMeasurementSystem
from video_preprocessor import PreprocessingConfig

# --- Configuration ---
VIDEO_PATH = 'test_video.mp4' # Replace with your video file
CAMERA_RADIUS = 150  # Camera rotation radius in cm
CALIBRATION_HEIGHT = 175  # Known height of the person in cm (optional)

# --- Preprocessing Configuration ---
preprocessing_config = PreprocessingConfig(
    enable_all_preprocessing=True, # Master switch for all preprocessing
    enable_visualization=False # Keep this false in a notebook to avoid window spam
)

# Check if video file exists
if not os.path.exists(VIDEO_PATH):
    print(f"Error: Video file not found at '{VIDEO_PATH}'")
    print("Please update the VIDEO_PATH variable to a valid file.")
else:
    print(f"Video file found: {VIDEO_PATH}")

print("Configuration and imports are loaded.")

## 2. Initialize and Run the System

Now we create an instance of the `BodyMeasurementSystem` and process the video. This step will perform:
1. Video frame extraction.
2. Frame preprocessing (if enabled).
3. Human pose detection.
4. 3D point cloud generation from each frame.
5. 3D model reconstruction.

In [ ]:
body_system = BodyMeasurementSystem(
    camera_radius=CAMERA_RADIUS,
    calibration_height=CALIBRATION_HEIGHT,
    preprocessing__config=preprocessing_config
)

# This can take a while depending on video length and preprocessing options
body_system.process_video(VIDEO_PATH)

## 3. Visualize the 3D Reconstructed Model

After processing, we can visualize the final 3D model. The visualization is interactive.

In [ ]:
# The visualize_3d_model function opens an interactive Open3D window
if body_system.model:
    print("Visualizing the 3D model. Close the window to continue.")
    body_system.visualize_3d_model()
else:
    print("No model was generated. Cannot visualize.")

## 4. Calculate and Display Body Measurements

With the 3D model, we can now calculate the key body measurements.

In [ ]:
if body_system.model:
    measurements = body_system.calculate_measurements()
    print("--- Calculated Body Measurements ---")
    for name, value in measurements.items():
        print(f"{name.replace('_', ' ').title()}: {value:.2f} cm")
    print("------------------------------------")
else:
    print("No model was generated. Cannot calculate measurements.")

## 5. Analyze Preprocessing Statistics

If preprocessing was enabled, we can inspect the statistics to understand the quality of the video frames and the performance of the preprocessing pipeline.

In [ ]:
stats = body_system.get_preprocessing_statistics()

if stats.get("preprocessing_enabled"):
    print("--- Preprocessing Statistics ---")
    body_system.print_preprocessing_summary()
    
    # Visualization of key statistics
    fig, axs = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('Preprocessing Quality Metrics', fontsize=16)
    
    # Quality Score
    if 'mean_quality' in stats:
        axs[0, 0].bar(['Mean Quality'], [stats['mean_quality']], yerr=[stats.get('std_quality', 0)], capsize=5)
        axs[0, 0].set_title('Overall Quality Score')
        axs[0, 0].set_ylabel('Score')
        axs[0, 0].set_ylim(0, 1)
        
    # Processing Time
    if 'avg_processing_time' in stats:
        axs[0, 1].bar(['Avg Time'], [stats['avg_processing_time']], yerr=[stats.get('std_processing_time', 0)], capsize=5)
        axs[0, 1].set_title('Frame Processing Time')
        axs[0, 1].set_ylabel('Seconds')
        
    # Brightness & Contrast
    if 'avg_brightness' in stats and 'avg_contrast' in stats:
        axs[1, 0].bar(['Brightness', 'Contrast'], [stats['avg_brightness'], stats['avg_contrast']], color=['gold', 'darkorange'])
        axs[1, 0].set_title('Image Characteristics')
        axs[1, 0].set_ylabel('Value')
        
    # Motion & Lighting
    if 'avg_motion_blur' in stats and 'avg_overexposure' in stats:
        axs[1, 1].bar(['Motion Blur', 'Overexposure', 'Underexposure'], 
                      [stats.get('avg_motion_blur',0), stats.get('avg_overexposure',0), stats.get('avg_underexposure',0)],
                      color=['dodgerblue', 'red', 'darkblue'])
        axs[1, 1].set_title('Potential Issues')
        axs[1, 1].set_ylabel('Score / Ratio')
    
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()
else:
    print("Preprocessing was not enabled.")

## 6. Conclusion

This notebook demonstrated the end-to-end process of calculating body measurements from a 360-degree video. The use of a comprehensive preprocessing pipeline and 3D reconstruction allows for robust measurements.